# E2 — Frequency-Only DCT Diagnostic v1

E2 trả lời một câu hỏi hẹp:

> **Frequency cue tự thân có đủ discriminative power cho Face PAD hay không?**

Đây là **diagnostic ablation**, không phải model proposed cuối cùng.

E2 cố ý **không có spatial backbone**. Input face crop dùng đúng split/cache/crop policy
đã freeze ở E1 v5.3, sau đó:

```text
RGB face crop
   ↓
same E1 train augmentation
   ↓
224×224 luminance
   ↓
2D DCT
   ↓
signed-log compression
   ↓
per-sample standardization
   ↓
Tiny Frequency CNN
   ↓
64-D frequency feature
   ↓
64 → 128 → 3 classifier
```

Classes giữ nguyên như E1:

```text
0 = Real
1 = Physical Spoof
2 = Digital Spoof
```

Binary PAD decision cũng giữ nguyên:

```text
d = real_logit - logsumexp([physical_spoof_logit, digital_spoof_logit])
REAL iff d >= calibrated_validation_threshold
```

## Fair-comparison rule

E2 **phải reuse trực tiếp** từ E1:

- exact Train / Validation / Test manifest;
- exact SCRFD bbox cache;
- crop factor `1.55×` cho SCRFD;
- CelebA fallback `1.50×`;
- TRAIN min-face filtering đã được E1 quyết định;
- same mild augmentation policy;
- batch size 128;
- AdamW + weight decay `1e-4`;
- warmup 2 + cosine;
- max 24 epochs;
- early-stop policy;
- Validation-only threshold calibration;
- seed 42.

Notebook này **không chạy lại SCRFD và không tự resample split**.

> Nếu artifact E1 là `preliminary`, E2 cũng là preliminary.
> Nếu chạy E1 `official`, E2 phải dùng đúng official cache/manifest tương ứng.


In [ ]:
# Dependencies
!pip install -q albumentations onnx onnxruntime-gpu onnxscript scikit-learn

import os
import glob
import json
import time
import math
import copy
import random
import hashlib
import shutil
import sys
from pathlib import PurePosixPath
from collections import Counter

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix

import onnx
import onnxruntime as ort

SAMPLE_SEED = 42
random.seed(SAMPLE_SEED)
np.random.seed(SAMPLE_SEED)
torch.manual_seed(SAMPLE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SAMPLE_SEED)

STRICT_DETERMINISM = False
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = STRICT_DETERMINISM
    torch.backends.cudnn.benchmark = not STRICT_DETERMINISM

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("ORT providers:", ort.get_available_providers())


## 1. Kaggle inputs

Attach two inputs:

1. **CelebA-Spoof dataset** — same dataset used by E1.
2. **E1 v5.3 artifacts** — either the saved output of the E1 Kaggle notebook or a private Kaggle Dataset containing those outputs.

The two E1 artifacts required by E2 are:

```text
celeba_scrfd_bbox_cache_v5_3_<mode>_seed42.json
celeba_spoof_<mode>_v5_3_edge_mnv3_small_seed42.npz
```

Recommended validation artifacts:

```text
celeba_spoof_<mode>_v5_3_preprocess_audit.json
mnv3s_e1_<mode>_v5_3_edge_run_config.json
```

The E1 `.pth` / `.onnx` are **not needed to train E2**, but must still be archived for E3 and deployment comparisons.


In [ ]:
# ---------------- User-editable Kaggle paths ----------------

# Must match the E1 artifacts you mounted.
E1_RUN_MODE = "preliminary"  # "preliminary" or "official"

# Leave blank to auto-discover exact filenames recursively under /kaggle/input.
# If multiple E1 artifact sets are mounted, set this explicitly.
E1_ARTIFACT_DIR = ""

# Same dataset root used by E1. Change only if your Kaggle dataset mount differs.
DATA_ROOT = (
    "/kaggle/input/datasets/attentionlayer241/"
    "celeba-spoof-for-face-antispoofing/"
    "CelebA_Spoof_/CelebA_Spoof"
)

TRAIN_JSON = os.path.join(DATA_ROOT, "metas/intra_test/train_label.json")
TEST_JSON = os.path.join(DATA_ROOT, "metas/intra_test/test_label.json")

EXPECTED_CACHE_NAME = (
    f"celeba_scrfd_bbox_cache_v5_3_{E1_RUN_MODE}_seed{SAMPLE_SEED}.json"
)
EXPECTED_MANIFEST_NAME = (
    f"celeba_spoof_{E1_RUN_MODE}_v5_3_edge_mnv3_small_seed42.npz"
)
EXPECTED_AUDIT_NAME = (
    f"celeba_spoof_{E1_RUN_MODE}_v5_3_preprocess_audit.json"
)
EXPECTED_E1_CONFIG_NAME = (
    f"mnv3s_e1_{E1_RUN_MODE}_v5_3_edge_run_config.json"
)


def find_exact_artifact(filename, explicit_dir=""):
    if explicit_dir:
        p = os.path.join(explicit_dir, filename)
        if os.path.exists(p):
            return p
        sub_matches = sorted(set(glob.glob(os.path.join(explicit_dir, "**", filename), recursive=True)))
        if len(sub_matches) == 1:
            return sub_matches[0]
        if len(sub_matches) > 1:
            raise RuntimeError(f"Multiple copies of {filename} found under {explicit_dir}:\n" + "\n".join(sub_matches))
        raise FileNotFoundError(f"Missing required artifact {filename} under {explicit_dir}")

    matches = glob.glob(
        os.path.join("/kaggle/input", "**", filename),
        recursive=True,
    )
    matches = sorted(set(matches))
    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} under /kaggle/input. "
            "Attach the E1 output/dataset or set E1_ARTIFACT_DIR."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple copies of {filename} found:\n"
            + "\n".join(matches)
            + "\nSet E1_ARTIFACT_DIR explicitly."
        )
    return matches[0]


def find_optional_artifact(filename, explicit_dir=""):
    try:
        return find_exact_artifact(filename, explicit_dir)
    except FileNotFoundError:
        return None


SCRFD_CACHE_PATH = find_exact_artifact(EXPECTED_CACHE_NAME, E1_ARTIFACT_DIR)
SPLIT_MANIFEST_PATH = find_exact_artifact(
    EXPECTED_MANIFEST_NAME, E1_ARTIFACT_DIR
)
PREPROCESS_AUDIT_PATH = find_optional_artifact(
    EXPECTED_AUDIT_NAME, E1_ARTIFACT_DIR
)
E1_RUN_CONFIG_PATH = find_optional_artifact(
    EXPECTED_E1_CONFIG_NAME, E1_ARTIFACT_DIR
)

if not os.path.exists(TRAIN_JSON) or not os.path.exists(TEST_JSON):
    raise FileNotFoundError(
        "CelebA-Spoof metadata not found at DATA_ROOT. "
        "Attach the same dataset used by E1 or edit DATA_ROOT."
    )

print("E1 run mode       :", E1_RUN_MODE)
print("SCRFD cache       :", SCRFD_CACHE_PATH)
print("Split manifest    :", SPLIT_MANIFEST_PATH)
print("Preprocess audit  :", PREPROCESS_AUDIT_PATH)
print("E1 run config     :", E1_RUN_CONFIG_PATH)
print("CelebA root       :", DATA_ROOT)


In [ ]:
# ---------------- Load and verify E1 artifacts ----------------

SCRFD_BBOX_EXPANSION_FACTOR = 1.55
CELEBA_FALLBACK_EXPANSION_FACTOR = 1.50
TRAIN_MIN_FACE_SIZE = 48

BBOX_JITTER_P = 0.20
BBOX_JITTER_SCALE = (0.95, 1.05)
BBOX_JITTER_TRANSLATE = 0.05

INPUT_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 4

with open(TRAIN_JSON, "r") as f:
    train_meta = json.load(f)
with open(TEST_JSON, "r") as f:
    test_meta = json.load(f)

manifest = np.load(SPLIT_MANIFEST_PATH, allow_pickle=False)
required_arrays = {"train_keys", "val_keys", "test_keys"}
missing_arrays = required_arrays - set(manifest.files)
if missing_arrays:
    raise RuntimeError(
        f"E1 manifest is missing arrays: {sorted(missing_arrays)}"
    )

train_keys = manifest["train_keys"].astype(str)
val_keys = manifest["val_keys"].astype(str)
test_keys = manifest["test_keys"].astype(str)

if "train_keys_requested" in manifest.files:
    train_keys_requested = manifest["train_keys_requested"].astype(str)
else:
    train_keys_requested = train_keys.copy()

with open(SCRFD_CACHE_PATH, "r") as f:
    cache_payload = json.load(f)

if not isinstance(cache_payload, dict) or "records" not in cache_payload:
    raise RuntimeError("Unexpected E1 SCRFD cache format.")

cache_schema = cache_payload.get("schema_version")
if cache_schema not in {
    "celeba_scrfd_bbox_cache_v5_3",
    "celeba_scrfd_bbox_cache_v5_2",
}:
    raise RuntimeError(
        f"Unexpected cache schema: {cache_schema}. "
        "Use the cache produced by the frozen detector-aligned E1."
    )

cache_policy = cache_payload.get("policy", {})
bbox_cache = cache_payload["records"]

# Enforce the frozen E1 crop contract.
if abs(float(cache_policy.get("scrfd_crop_factor", -1)) - 1.55) > 1e-9:
    raise RuntimeError(f"Unexpected SCRFD crop factor: {cache_policy}")
if abs(float(cache_policy.get("celeba_fallback_crop_factor", -1)) - 1.50) > 1e-9:
    raise RuntimeError(f"Unexpected CelebA fallback factor: {cache_policy}")
if int(cache_policy.get("train_min_face_px", -1)) != 48:
    raise RuntimeError(f"Unexpected train min-face policy: {cache_policy}")

selected_keys = list(dict.fromkeys(
    list(map(str, train_keys_requested))
    + list(map(str, val_keys))
    + list(map(str, test_keys))
))
missing_cache = [k for k in selected_keys if k not in bbox_cache]
if missing_cache:
    raise RuntimeError(
        f"{len(missing_cache)} E1 manifest samples are missing from the SCRFD cache."
    )


def label_of(meta_value):
    if len(meta_value) <= 40:
        raise ValueError(f"Metadata entry too short: len={len(meta_value)}")
    raw = int(meta_value[40])
    if raw == 0:
        return 0
    if raw in [1, 2, 3, 4, 5, 6, 7]:
        return 1
    return 2


CLASS_NAMES = {
    0: "Real",
    1: "Physical Spoof",
    2: "Digital Spoof",
}


def class_counts(keys, meta):
    y = np.array([label_of(meta[str(k)]) for k in keys], dtype=np.int64)
    return {c: int((y == c).sum()) for c in [0, 1, 2]}


def split_fingerprint(keys):
    payload = "\n".join(sorted(map(str, keys))).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def selected_cache_fingerprint(keys, records):
    h = hashlib.sha256()
    for key in sorted(map(str, keys)):
        h.update(key.encode("utf-8"))
        h.update(b"\0")
        h.update(
            json.dumps(
                records[key],
                sort_keys=True,
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(b"\n")
    return h.hexdigest()


split_fingerprints = {
    "train": split_fingerprint(train_keys),
    "val": split_fingerprint(val_keys),
    "test": split_fingerprint(test_keys),
}
cache_fingerprint = selected_cache_fingerprint(
    selected_keys,
    bbox_cache,
)

# Optional cross-check against E1 run config.
if E1_RUN_CONFIG_PATH:
    with open(E1_RUN_CONFIG_PATH, "r") as f:
        e1_config = json.load(f)

    expected_splits = e1_config.get("split_fingerprints")
    if expected_splits:
        for split_name in ["train", "val", "test"]:
            if expected_splits.get(split_name) != split_fingerprints[split_name]:
                raise RuntimeError(
                    f"{split_name} fingerprint differs from E1 run config."
                )

    expected_cache_fp = (
        e1_config.get("bbox_cache_selected_fingerprint")
        or e1_config.get("bbox_cache_sha256")
    )
    if expected_cache_fp and expected_cache_fp != cache_fingerprint:
        raise RuntimeError(
            "SCRFD cache fingerprint differs from the E1 run config."
        )

print("=== E1 artifact verification PASS ===")
print("Train:", len(train_keys), class_counts(train_keys, train_meta))
print("Val  :", len(val_keys), class_counts(val_keys, train_meta))
print("Test :", len(test_keys), class_counts(test_keys, test_meta))
print("Split fingerprints:", {k: v[:16] + "..." for k, v in split_fingerprints.items()})
print("Cache fingerprint :", cache_fingerprint[:16] + "...")
print("Cache schema      :", cache_schema)
print("Cache policy      :", cache_policy)


## 2. Frequency preprocessing

The project design specifies:

```text
RGB face
→ luminance/grayscale
→ 2D DCT
→ sign(C) * log(1 + |C|)
→ stable normalization
```

This notebook makes the final normalization choice explicit:

```text
signed-log DCT
→ per-sample z-score over the full 224×224 DCT map
```

No frequency band is manually removed or emphasized in E2. The Tiny CNN sees the
full DCT map so E2 tests whether it can learn discriminative patterns across the spectrum.

This is an **engineering implementation choice** for E2, not an assumption that
high-frequency energy itself means spoof.


In [ ]:
# ---------------- Frozen crop helpers copied from E1 semantics ----------------

def bbox_wh(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([x2 - x1, y2 - y1], dtype=np.float64)


def runtime_crop_bgr(img, bbox_xyxy, bbox_expansion_factor):
    original_height, original_width = img.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        raise ValueError(f"Invalid xyxy bbox: {bbox_xyxy}")

    max_dim = max(w, h)
    center_x = x1 + w / 2.0
    center_y = y1 + h / 2.0

    x_start = int(center_x - max_dim * bbox_expansion_factor / 2.0)
    y_start = int(center_y - max_dim * bbox_expansion_factor / 2.0)
    crop_size = int(max_dim * bbox_expansion_factor)
    if crop_size <= 0:
        raise ValueError(f"Invalid crop_size={crop_size}")

    crop_x1 = max(0, x_start)
    crop_y1 = max(0, y_start)
    crop_x2 = min(original_width, x_start + crop_size)
    crop_y2 = min(original_height, y_start + crop_size)

    top_pad = int(max(0, -y_start))
    left_pad = int(max(0, -x_start))
    bottom_pad = int(max(0, (y_start + crop_size) - original_height))
    right_pad = int(max(0, (x_start + crop_size) - original_width))

    if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
        raise RuntimeError(f"Empty crop intersection for bbox={bbox_xyxy}")

    cropped_img = img[crop_y1:crop_y2, crop_x1:crop_x2, :]
    result = cv2.copyMakeBorder(
        cropped_img,
        top_pad,
        bottom_pad,
        left_pad,
        right_pad,
        cv2.BORDER_REFLECT_101,
    )

    if result.shape[0] != crop_size or result.shape[1] != crop_size:
        result = cv2.resize(
            result,
            (crop_size, crop_size),
            interpolation=cv2.INTER_AREA,
        )
    return result


def read_cached_face_crop_rgb(
    img_path,
    cache_record,
    bbox_jitter=False,
    bbox_jitter_p=BBOX_JITTER_P,
    jitter_scale=BBOX_JITTER_SCALE,
    jitter_translate=BBOX_JITTER_TRANSLATE,
):
    image_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {img_path}")

    if cache_record.get("status") == "INVALID":
        raise RuntimeError(
            f"Invalid cached preprocessing record for {img_path}: {cache_record}"
        )

    x1, y1, x2, y2 = map(float, cache_record["bbox_xyxy"])
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        raise ValueError(f"Non-positive cached bbox for {img_path}")

    if bbox_jitter and random.random() < float(bbox_jitter_p):
        base = max(w, h)
        cx = x1 + w / 2.0
        cy = y1 + h / 2.0

        scale = random.uniform(
            float(jitter_scale[0]),
            float(jitter_scale[1]),
        )
        cx += random.uniform(-jitter_translate, jitter_translate) * base
        cy += random.uniform(-jitter_translate, jitter_translate) * base
        w *= scale
        h *= scale

        x1 = cx - w / 2.0
        y1 = cy - h / 2.0
        x2 = x1 + w
        y2 = y1 + h

    crop_bgr = runtime_crop_bgr(
        image_bgr,
        (x1, y1, x2, y2),
        bbox_expansion_factor=float(cache_record["crop_factor"]),
    )
    return cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)


def resize_rgb_to_input(img_rgb, size=INPUT_SIZE):
    old_h, old_w = img_rgb.shape[:2]
    ratio = float(size) / max(old_h, old_w)
    scaled_h = max(1, int(old_h * ratio))
    scaled_w = max(1, int(old_w * ratio))
    interpolation = cv2.INTER_LANCZOS4 if ratio > 1.0 else cv2.INTER_AREA

    img = cv2.resize(
        img_rgb,
        (scaled_w, scaled_h),
        interpolation=interpolation,
    )

    delta_w = size - scaled_w
    delta_h = size - scaled_h
    top, bottom = delta_h // 2, delta_h - delta_h // 2
    left, right = delta_w // 2, delta_w - delta_w // 2

    return cv2.copyMakeBorder(
        img,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_REFLECT_101,
    )


def rgb_to_luminance_float(img_rgb):
    rgb = img_rgb.astype(np.float32) / 255.0
    # Standard RGB luminance weighting.
    return (
        0.299 * rgb[..., 0]
        + 0.587 * rgb[..., 1]
        + 0.114 * rgb[..., 2]
    ).astype(np.float32)


DCT_EPS = 1e-6


def luminance_to_dct_map(luma):
    if luma.shape != (INPUT_SIZE, INPUT_SIZE):
        raise ValueError(f"Expected {(INPUT_SIZE, INPUT_SIZE)}, got {luma.shape}")

    coeff = cv2.dct(np.ascontiguousarray(luma, dtype=np.float32))
    coeff = np.sign(coeff) * np.log1p(np.abs(coeff))

    mean = float(coeff.mean())
    std = float(coeff.std())
    coeff = (coeff - mean) / max(std, DCT_EPS)

    return coeff[None, ...].astype(np.float32)


def rgb_to_frequency_tensor(img_rgb):
    img_rgb = resize_rgb_to_input(img_rgb, INPUT_SIZE)
    luma = rgb_to_luminance_float(img_rgb)
    dct_map = luminance_to_dct_map(luma)
    return torch.from_numpy(np.ascontiguousarray(dct_map))


In [ ]:
# ---------------- Dataset and DataLoaders ----------------

train_augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(
        brightness_limit=0.10,
        contrast_limit=0.10,
        p=0.30,
    ),
])


class CelebASpoofFrequencyDataset(Dataset):
    def __init__(
        self,
        root_dir,
        json_path,
        keys,
        bbox_cache,
        augment_transform=None,
        bbox_jitter=False,
        bbox_jitter_p=0.0,
    ):
        self.root_dir = root_dir
        with open(json_path, "r") as f:
            self.meta = json.load(f)

        self.keys = list(map(str, keys))
        self.bbox_cache = bbox_cache
        self.augment_transform = augment_transform
        self.bbox_jitter = bool(bbox_jitter)
        self.bbox_jitter_p = float(bbox_jitter_p)

        missing_meta = [k for k in self.keys if k not in self.meta]
        missing_cache = [k for k in self.keys if k not in self.bbox_cache]
        if missing_meta:
            raise KeyError(f"{len(missing_meta)} keys missing metadata.")
        if missing_cache:
            raise KeyError(f"{len(missing_cache)} keys missing bbox cache.")

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        rel_path = self.keys[idx]
        img_path = os.path.join(self.root_dir, rel_path)
        rec = self.bbox_cache[rel_path]

        crop_rgb = read_cached_face_crop_rgb(
            img_path,
            rec,
            bbox_jitter=self.bbox_jitter,
            bbox_jitter_p=self.bbox_jitter_p,
        )

        # Keep E1 train augmentation before frequency transform.
        if self.augment_transform is not None:
            crop_rgb = self.augment_transform(image=crop_rgb)["image"]

        x = rgb_to_frequency_tensor(crop_rgb)
        y = label_of(self.meta[rel_path])
        return x, torch.tensor(y, dtype=torch.long)


train_dataset = CelebASpoofFrequencyDataset(
    DATA_ROOT,
    TRAIN_JSON,
    train_keys,
    bbox_cache,
    augment_transform=train_augment,
    bbox_jitter=True,
    bbox_jitter_p=BBOX_JITTER_P,
)
val_dataset = CelebASpoofFrequencyDataset(
    DATA_ROOT,
    TRAIN_JSON,
    val_keys,
    bbox_cache,
    augment_transform=None,
    bbox_jitter=False,
)
test_dataset = CelebASpoofFrequencyDataset(
    DATA_ROOT,
    TEST_JSON,
    test_keys,
    bbox_cache,
    augment_transform=None,
    bbox_jitter=False,
)

assert len(train_dataset) == len(train_keys)
assert len(val_dataset) == len(val_keys)
assert len(test_dataset) == len(test_keys)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(SAMPLE_SEED)

common_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=False,
    worker_init_fn=seed_worker,
)

train_drop_last = (len(train_dataset) % BATCH_SIZE == 1)
train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=loader_generator,
    drop_last=train_drop_last,
    **common_loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **common_loader_kwargs,
)
test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **common_loader_kwargs,
)

print("DataLoaders ready:")
print("  Train:", len(train_dataset))
print("  Val  :", len(val_dataset))
print("  Test :", len(test_dataset))


In [ ]:
# Optional DCT sanity visualization
RUN_DCT_SANITY = False
DCT_SANITY_SAMPLES = 4

if RUN_DCT_SANITY:
    fig, axes = plt.subplots(
        DCT_SANITY_SAMPLES,
        2,
        figsize=(8, 3 * DCT_SANITY_SAMPLES),
    )

    for row in range(DCT_SANITY_SAMPLES):
        key = str(train_keys[row])
        img_path = os.path.join(DATA_ROOT, key)
        rec = bbox_cache[key]

        crop = read_cached_face_crop_rgb(
            img_path,
            rec,
            bbox_jitter=False,
        )
        resized = resize_rgb_to_input(crop)
        freq = rgb_to_frequency_tensor(resized).numpy()[0]

        axes[row, 0].imshow(resized)
        axes[row, 0].set_title(
            f"{CLASS_NAMES[label_of(train_meta[key])]}\n{rec['bbox_source']}"
        )
        axes[row, 0].axis("off")

        vmax = np.percentile(np.abs(freq), 99)
        axes[row, 1].imshow(
            freq,
            cmap="gray",
            vmin=-vmax,
            vmax=vmax,
        )
        axes[row, 1].set_title("signed-log DCT, z-scored")
        axes[row, 1].axis("off")

    plt.tight_layout()
    plt.show()


## 3. E2 model

The frequency branch is deliberately small and reusable in E3.

```text
1×224×224 DCT map
   ↓
3×3 Conv, 1→32, stride 2
   ↓
Depthwise 3×3, 32, stride 2
Pointwise 1×1, 32→32
   ↓
Depthwise 3×3, 32, stride 2
Pointwise 1×1, 32→64
   ↓
Global Average Pool
   ↓
Linear 64→64
   ↓
64-D frequency feature
```

E2-only classifier:

```text
64 → 128 → 3
```

The `FrequencyBranch` module is saved separately so E3 can reuse the exact architecture
and optionally its E2 weights. Whether E3 initializes from E2 weights must be decided
and documented before E3 training; it should not happen silently.


In [ ]:
# ---------------- E2 Frequency-Only Model ----------------

NUM_CLASSES = 3
FREQ_DIM = 64
HIDDEN_DIM = 128
DROPOUT_RATE = 0.20
LABEL_SMOOTHING = 0.10

EPOCHS = 24
FREQ_LR = 1e-4
WEIGHT_DECAY = 1e-4

WARMUP_EPOCHS = 2
WARMUP_START_FACTOR = 0.20
MIN_LR_FACTOR = 0.10

EARLY_STOP_PATIENCE = 4
EARLY_STOP_MIN_DELTA = 1e-4
EARLY_STOP_MIN_EPOCHS = 10
GRAD_CLIP_NORM = 5.0

PROTOCOL_VERSION = "e2_frequency_only_dct_v1"


class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                in_ch,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_ch,
                bias=False,
            ),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class FrequencyBranch(nn.Module):
    def __init__(self, out_dim=FREQ_DIM):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            DepthwiseSeparableBlock(32, 32, stride=2),
            DepthwiseSeparableBlock(32, 64, stride=2),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, out_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.proj(self.features(x))


class FrequencyOnlyPAD(nn.Module):
    def __init__(
        self,
        freq_dim=FREQ_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT_RATE,
    ):
        super().__init__()
        self.frequency_branch = FrequencyBranch(freq_dim)
        self.classifier = nn.Sequential(
            nn.Linear(freq_dim, hidden_dim),
            nn.Hardswish(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        freq_feat = self.frequency_branch(x)
        return self.classifier(freq_feat)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FrequencyOnlyPAD().to(device)

MODEL_PARAMETER_COUNT = int(sum(p.numel() for p in model.parameters()))
BRANCH_PARAMETER_COUNT = int(
    sum(p.numel() for p in model.frequency_branch.parameters())
)

criterion = nn.CrossEntropyLoss(
    weight=torch.ones(NUM_CLASSES, device=device),
    label_smoothing=LABEL_SMOOTHING,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=FREQ_LR,
    weight_decay=WEIGHT_DECAY,
)


def lr_multiplier(epoch_index: int) -> float:
    epoch_index = int(epoch_index)

    if WARMUP_EPOCHS > 0 and epoch_index < WARMUP_EPOCHS:
        return WARMUP_START_FACTOR + (
            1.0 - WARMUP_START_FACTOR
        ) * (epoch_index / WARMUP_EPOCHS)

    cosine_epochs = max(1, EPOCHS - WARMUP_EPOCHS)
    denom = max(1, cosine_epochs - 1)
    progress = min(
        1.0,
        max(0.0, (epoch_index - WARMUP_EPOCHS) / denom),
    )
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_multiplier,
)

amp_enabled = device.type == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

print(model)
print("Protocol:", PROTOCOL_VERSION)
print("Device:", device)
print(f"Total parameters : {MODEL_PARAMETER_COUNT:,}")
print(f"Branch parameters: {BRANCH_PARAMETER_COUNT:,}")
print("LR:", FREQ_LR)


In [ ]:
# ---------------- PAD metrics and threshold calibration ----------------

def binary_pad_score_from_logits(logits_np):
    logits = np.asarray(logits_np, dtype=np.float64)
    real = logits[:, 0]
    spoof = logits[:, 1:]

    m = np.max(spoof, axis=1)
    spoof_lse = m + np.log(
        np.exp(spoof[:, 0] - m)
        + np.exp(spoof[:, 1] - m)
    )
    return real - spoof_lse


def probability_from_logit_threshold(logit_threshold):
    return float(1.0 / (1.0 + np.exp(-float(logit_threshold))))


def metrics_from_scores(y3, d, threshold):
    y3 = np.asarray(y3, dtype=np.int64)
    d = np.asarray(d, dtype=np.float64)

    is_real = (y3 == 0)
    is_attack = ~is_real
    pred_real = d >= threshold

    bpcer = (
        float(np.mean(~pred_real[is_real]))
        if np.any(is_real) else float("nan")
    )
    apcer = (
        float(np.mean(pred_real[is_attack]))
        if np.any(is_attack) else float("nan")
    )
    acer = 0.5 * (apcer + bpcer)

    y_binary = is_real.astype(np.int64)
    try:
        auc = float(roc_auc_score(y_binary, d))
    except ValueError:
        auc = float("nan")

    accuracy = float(np.mean(pred_real == is_real))
    cm = confusion_matrix(
        y_binary,
        pred_real.astype(np.int64),
        labels=[0, 1],
    ).tolist()

    return {
        "Accuracy": accuracy,
        "APCER": apcer,
        "BPCER": bpcer,
        "ACER": acer,
        "AUC": auc,
        "confusion_matrix": cm,
    }


def calibrate_threshold(y3, d):
    y3 = np.asarray(y3, dtype=np.int64)
    d = np.asarray(d, dtype=np.float64)

    unique_scores = np.unique(d)
    if len(unique_scores) == 1:
        candidates = unique_scores
    else:
        mids = (unique_scores[:-1] + unique_scores[1:]) / 2.0
        candidates = np.concatenate([
            [unique_scores[0] - 1e-6],
            mids,
            [unique_scores[-1] + 1e-6],
        ])

    is_real = (y3 == 0)
    is_attack = ~is_real

    # Vectorized threshold sweep.
    pred_real = d[:, None] >= candidates[None, :]
    bpcer = np.mean(~pred_real[is_real], axis=0)
    apcer = np.mean(pred_real[is_attack], axis=0)
    acer = 0.5 * (apcer + bpcer)

    # Primary objective: ACER. Tie-break: lower APCER, then lower BPCER.
    best_idx = np.lexsort((bpcer, apcer, acer))[0]
    threshold = float(candidates[best_idx])

    result = metrics_from_scores(y3, d, threshold)
    result["LogitThreshold"] = threshold
    result["PredictorProbabilityThreshold"] = (
        probability_from_logit_threshold(threshold)
    )
    return result


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device, calibrate=False, threshold=None):
    model.eval()

    total_loss = 0.0
    total_count = 0
    total_correct_3class = 0

    all_logits = []
    all_labels = []

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda"),
        ):
            logits = model(x)
            loss = criterion(logits, y)

        n = y.size(0)
        total_loss += float(loss.item()) * n
        total_count += n
        total_correct_3class += int((logits.argmax(1) == y).sum().item())

        all_logits.append(logits.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)
    d = binary_pad_score_from_logits(logits_np)

    if calibrate:
        result = calibrate_threshold(labels_np, d)
    else:
        if threshold is None:
            raise ValueError("threshold required when calibrate=False")
        result = metrics_from_scores(labels_np, d, threshold)
        result["LogitThreshold"] = float(threshold)
        result["PredictorProbabilityThreshold"] = (
            probability_from_logit_threshold(threshold)
        )

    result["Loss"] = total_loss / max(1, total_count)
    result["Accuracy3Class"] = total_correct_3class / max(1, total_count)
    return result


## 4. Training

E2 keeps the E1 schedule but all trainable parameters are new, so the frequency-only
network uses the E1 **head LR** (`1e-4`) for all of its parameters.

Checkpoint selection:

```text
Validation ACER
```

Threshold:

```text
calibrate on Validation only
```

Held-out Test remains disabled by default until the E2 architecture and training
protocol are frozen.


In [ ]:
# ---------------- Training / checkpointing ----------------

RUN_NAME = f"e2_freq_only_{E1_RUN_MODE}_dct_v1"

CHECKPOINT_PATH = f"{RUN_NAME}_checkpoint_last.pth"
BEST_MODEL_PATH = f"{RUN_NAME}_best.pth"
BEST_META_PATH = f"{RUN_NAME}_best_meta.json"
HISTORY_PATH = f"{RUN_NAME}_training_history.json"
RUN_CONFIG_PATH = f"{RUN_NAME}_run_config.json"
BRANCH_BEST_PATH = f"{RUN_NAME}_frequency_branch_best.pth"
BRANCH_SPEC_PATH = f"{RUN_NAME}_frequency_branch_spec.json"

CHECKPOINT_INPUT_PATH = ""


def cpu_state_dict(module):
    return {
        k: v.detach().cpu().clone()
        for k, v in module.state_dict().items()
    }


RUN_CONFIG = {
    "protocol_version": PROTOCOL_VERSION,
    "experiment": "E2_frequency_only",
    "research_question": "Can frequency cues alone discriminate PAD classes?",
    "e1_run_mode": E1_RUN_MODE,
    "e1_split_manifest": os.path.basename(SPLIT_MANIFEST_PATH),
    "e1_scrfd_cache": os.path.basename(SCRFD_CACHE_PATH),
    "e1_split_fingerprints": split_fingerprints,
    "e1_cache_selected_fingerprint": cache_fingerprint,

    "input_size": INPUT_SIZE,
    "input_type": "1-channel signed-log DCT map",
    "dct_source": "RGB crop -> luminance",
    "luminance_weights_rgb": [0.299, 0.587, 0.114],
    "dct_dynamic_range": "sign(C)*log1p(abs(C))",
    "dct_normalization": "per-sample z-score over full map",
    "frequency_mask": "none",

    "frequency_dim": FREQ_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_classes": NUM_CLASSES,
    "model_parameter_count": MODEL_PARAMETER_COUNT,
    "frequency_branch_parameter_count": BRANCH_PARAMETER_COUNT,

    "optimizer": "AdamW",
    "lr": FREQ_LR,
    "weight_decay": WEIGHT_DECAY,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "scheduler": "WarmupCosineLambdaLR",
    "warmup_epochs": WARMUP_EPOCHS,
    "warmup_start_factor": WARMUP_START_FACTOR,
    "min_lr_factor": MIN_LR_FACTOR,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
    "early_stop_min_epochs": EARLY_STOP_MIN_EPOCHS,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "dropout": DROPOUT_RATE,
    "label_smoothing": LABEL_SMOOTHING,
    "class_weights": [1.0, 1.0, 1.0],
    "sample_seed": SAMPLE_SEED,

    "augmentation": {
        "horizontal_flip_p": 0.5,
        "brightness_limit": 0.10,
        "contrast_limit": 0.10,
        "appearance_p": 0.30,
        "bbox_jitter_p": BBOX_JITTER_P,
        "bbox_scale_jitter": list(BBOX_JITTER_SCALE),
        "bbox_translate_fraction": BBOX_JITTER_TRANSLATE,
    },
    "score_definition": "real_minus_logsumexp_spoof",
}

with open(RUN_CONFIG_PATH, "w") as f:
    json.dump(RUN_CONFIG, f, indent=2)

history = {
    "train_loss": [],
    "train_acc_3class": [],
    "val_loss": [],
    "val_acc_3class": [],
    "val_apcer": [],
    "val_bpcer": [],
    "val_acer": [],
    "val_auc": [],
    "val_logit_threshold": [],
    "lr": [],
}

best_val_acer = float("inf")
best_epoch = -1
best_model_state = None
best_val_metrics = None
best_logit_threshold = None
epochs_without_improvement = 0
start_epoch = 0


def save_best_artifacts(
    model,
    best_state,
    best_epoch,
    best_val_metrics,
    best_logit_threshold,
):
    torch.save(best_state, BEST_MODEL_PATH)

    # Save branch separately for E3 reuse.
    branch_state = {
        k.replace("frequency_branch.", "", 1): v
        for k, v in best_state.items()
        if k.startswith("frequency_branch.")
    }
    torch.save(branch_state, BRANCH_BEST_PATH)

    branch_spec = {
        "class": "FrequencyBranch",
        "protocol_version": PROTOCOL_VERSION,
        "input": [1, INPUT_SIZE, INPUT_SIZE],
        "output_dim": FREQ_DIM,
        "dct": RUN_CONFIG["dct_dynamic_range"],
        "normalization": RUN_CONFIG["dct_normalization"],
        "frequency_mask": "none",
        "parameter_count": BRANCH_PARAMETER_COUNT,
    }
    with open(BRANCH_SPEC_PATH, "w") as f:
        json.dump(branch_spec, f, indent=2)

    best_meta = {
        "protocol_version": PROTOCOL_VERSION,
        "best_epoch": int(best_epoch),
        "model_parameter_count": MODEL_PARAMETER_COUNT,
        "frequency_branch_parameter_count": BRANCH_PARAMETER_COUNT,
        "calibrated_logit_threshold": float(best_logit_threshold),
        "calibrated_probability_threshold": (
            probability_from_logit_threshold(best_logit_threshold)
        ),
        "validation_metrics": {
            k: (
                float(v)
                if isinstance(v, (int, float, np.floating))
                else v
            )
            for k, v in best_val_metrics.items()
        },
        "e1_split_fingerprints": split_fingerprints,
        "e1_cache_selected_fingerprint": cache_fingerprint,
        "frequency_branch_artifact": BRANCH_BEST_PATH,
    }
    with open(BEST_META_PATH, "w") as f:
        json.dump(best_meta, f, indent=2)


# Optional resume.
resume_source = None
if os.path.exists(CHECKPOINT_PATH):
    resume_source = CHECKPOINT_PATH
elif CHECKPOINT_INPUT_PATH and os.path.exists(CHECKPOINT_INPUT_PATH):
    shutil.copy(CHECKPOINT_INPUT_PATH, CHECKPOINT_PATH)
    resume_source = CHECKPOINT_PATH

if resume_source:
    ckpt = torch.load(resume_source, map_location=device)

    if ckpt.get("run_config") != RUN_CONFIG:
        raise RuntimeError(
            "Checkpoint RUN_CONFIG differs from current E2 protocol. "
            "Do not resume across changed DCT/split/cache/training settings."
        )

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    try:
        scaler.load_state_dict(ckpt["scaler_state"])
    except Exception:
        pass

    start_epoch = int(ckpt["epoch"]) + 1
    history = ckpt["history"]
    best_val_acer = float(ckpt["best_val_acer"])
    best_epoch = int(ckpt["best_epoch"])
    best_model_state = ckpt.get("best_model_state")
    best_val_metrics = ckpt.get("best_val_metrics")
    best_logit_threshold = ckpt.get("best_logit_threshold")
    epochs_without_improvement = int(
        ckpt.get("epochs_without_improvement", 0)
    )

    print(
        f"✅ Resumed E2 from epoch {start_epoch + 1}/{EPOCHS}; "
        f"best Val ACER={best_val_acer*100:.3f}%"
    )


for epoch in range(start_epoch, EPOCHS):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_count = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
    )

    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda"),
        ):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP_NORM,
        )
        scaler.step(optimizer)
        scaler.update()

        n = y.size(0)
        running_loss += float(loss.item()) * n
        running_correct += int((logits.argmax(1) == y).sum().item())
        running_count += n

        pbar.set_postfix(
            loss=f"{running_loss/max(1,running_count):.4f}",
            acc=f"{100*running_correct/max(1,running_count):.2f}%",
        )

    train_loss = running_loss / max(1, running_count)
    train_acc = running_correct / max(1, running_count)

    val_metrics = evaluate_loader(
        model,
        val_loader,
        criterion,
        device,
        calibrate=True,
    )

    current_acer = float(val_metrics["ACER"])
    current_threshold = float(val_metrics["LogitThreshold"])

    history["train_loss"].append(train_loss)
    history["train_acc_3class"].append(train_acc)
    history["val_loss"].append(float(val_metrics["Loss"]))
    history["val_acc_3class"].append(float(val_metrics["Accuracy3Class"]))
    history["val_apcer"].append(float(val_metrics["APCER"]))
    history["val_bpcer"].append(float(val_metrics["BPCER"]))
    history["val_acer"].append(current_acer)
    history["val_auc"].append(float(val_metrics["AUC"]))
    history["val_logit_threshold"].append(current_threshold)
    history["lr"].append(float(optimizer.param_groups[0]["lr"]))

    best_before = best_val_acer

    if current_acer < best_val_acer - 1e-12:
        best_val_acer = current_acer
        best_epoch = epoch
        best_model_state = cpu_state_dict(model)
        best_val_metrics = copy.deepcopy(val_metrics)
        best_logit_threshold = current_threshold

        save_best_artifacts(
            model,
            best_model_state,
            best_epoch,
            best_val_metrics,
            best_logit_threshold,
        )

        print(
            f"⭐ New best E2: epoch={epoch+1}, "
            f"Val ACER={100*current_acer:.3f}%, "
            f"APCER={100*val_metrics['APCER']:.3f}%, "
            f"BPCER={100*val_metrics['BPCER']:.3f}%, "
            f"AUC={100*val_metrics['AUC']:.3f}%, "
            f"logit_thr={current_threshold:.6f}"
        )

    if current_acer < best_before - EARLY_STOP_MIN_DELTA:
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    scheduler.step()

    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "history": history,
        "best_val_acer": best_val_acer,
        "best_epoch": best_epoch,
        "best_model_state": best_model_state,
        "best_val_metrics": best_val_metrics,
        "best_logit_threshold": best_logit_threshold,
        "epochs_without_improvement": epochs_without_improvement,
        "run_config": RUN_CONFIG,
    }
    torch.save(checkpoint, CHECKPOINT_PATH)

    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

    print(
        f"Epoch {epoch+1}: "
        f"train_loss={train_loss:.4f}, "
        f"train_acc={100*train_acc:.2f}%, "
        f"val_ACER={100*current_acer:.3f}%, "
        f"val_AUC={100*val_metrics['AUC']:.3f}%"
    )

    if (
        epoch + 1 >= EARLY_STOP_MIN_EPOCHS
        and epochs_without_improvement >= EARLY_STOP_PATIENCE
    ):
        print("Early stopping.")
        break


if best_model_state is None:
    raise RuntimeError("No best E2 checkpoint was produced.")

model.load_state_dict(best_model_state)

# Training history plot.
epochs_axis = np.arange(1, len(history["val_acer"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_axis, np.array(history["val_acer"]) * 100, label="Val ACER")
plt.plot(epochs_axis, np.array(history["val_apcer"]) * 100, label="Val APCER")
plt.plot(epochs_axis, np.array(history["val_bpcer"]) * 100, label="Val BPCER")
plt.xlabel("Epoch")
plt.ylabel("%")
plt.title("E2 Frequency-Only Validation PAD Metrics")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(f"{RUN_NAME}_training_history.png", dpi=160)
plt.show()

print("Best validation ACER:", f"{100*best_val_acer:.3f}%")
print("Best epoch:", best_epoch + 1)
print("Locked logit threshold:", best_logit_threshold)
print(
    "Probability threshold:",
    probability_from_logit_threshold(best_logit_threshold),
)
print("Best model:", BEST_MODEL_PATH)
print("Reusable frequency branch:", BRANCH_BEST_PATH)


In [ ]:
# ---------------- Held-out Test: keep gated until E2 is frozen ----------------

RUN_HELDOUT_TEST = False

with open(BEST_META_PATH, "r") as f:
    best_meta = json.load(f)

locked_logit_threshold = float(
    best_meta["calibrated_logit_threshold"]
)

if RUN_HELDOUT_TEST:
    test_results = evaluate_loader(
        model,
        test_loader,
        criterion,
        device,
        calibrate=False,
        threshold=locked_logit_threshold,
    )

    print("E2 CelebA-Spoof Held-out Test")
    for key in ["Accuracy", "APCER", "BPCER", "ACER", "AUC"]:
        print(f"  {key:8s}: {100*test_results[key]:.3f}%")
    print("  CM:", test_results["confusion_matrix"])

    with open(f"{RUN_NAME}_heldout_test_results.json", "w") as f:
        json.dump({
            "protocol_version": PROTOCOL_VERSION,
            "locked_validation_logit_threshold": locked_logit_threshold,
            "locked_validation_probability_threshold": (
                probability_from_logit_threshold(locked_logit_threshold)
            ),
            "metrics": test_results,
            "e1_split_fingerprints": split_fingerprints,
            "e1_cache_selected_fingerprint": cache_fingerprint,
        }, f, indent=2)
else:
    print(
        "Held-out Test is disabled. "
        "Enable only after E2 architecture/training protocol is frozen."
    )


## 5. Export

E2 ONNX consumes the **precomputed DCT map**, shape:

```text
[N, 1, 224, 224]
```

DCT/luminance preprocessing remains outside the ONNX graph in this diagnostic.

The important E2 research artifact for E3 is also:

```text
*_frequency_branch_best.pth
```

which contains only the reusable `FrequencyBranch` weights.


In [ ]:
# ---------------- Export FP32 ONNX + parity ----------------

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device,
        weights_only=True,
    )
)
model.eval()

onnx_path = f"{RUN_NAME}_best.onnx"

dummy = torch.randn(
    1,
    1,
    INPUT_SIZE,
    INPUT_SIZE,
    device=device,
)

with torch.no_grad():
    torch_out = model(dummy).cpu().numpy()

torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=["frequency_map"],
    output_names=["logits"],
    dynamic_axes={
        "frequency_map": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    opset_version=18,
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

session = ort.InferenceSession(
    onnx_path,
    providers=["CPUExecutionProvider"],
)
onnx_out = session.run(
    None,
    {"frequency_map": dummy.detach().cpu().numpy()},
)[0]

max_abs_diff = float(np.max(np.abs(torch_out - onnx_out)))
if max_abs_diff >= 1e-4:
    raise RuntimeError(
        f"PyTorch↔ONNX parity failed: max |Δ|={max_abs_diff}"
    )

onnx_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)

# Host-CPU diagnostic only.
bench_x = np.random.randn(
    1, 1, INPUT_SIZE, INPUT_SIZE
).astype(np.float32)

for _ in range(20):
    session.run(None, {"frequency_map": bench_x})

times_ms = []
for _ in range(100):
    t0 = time.perf_counter()
    session.run(None, {"frequency_map": bench_x})
    times_ms.append((time.perf_counter() - t0) * 1000.0)

cpu_latency = {
    "mean_ms": float(np.mean(times_ms)),
    "median_ms": float(np.median(times_ms)),
    "p90_ms": float(np.percentile(times_ms, 90)),
}

runtime_config = {
    "protocol_version": PROTOCOL_VERSION,
    "experiment": "E2_frequency_only",
    "onnx_model": onnx_path,
    "onnx_input": "frequency_map",
    "onnx_input_shape": ["batch", 1, INPUT_SIZE, INPUT_SIZE],
    "onnx_fp32_size_mb": float(onnx_size_mb),
    "model_parameter_count": MODEL_PARAMETER_COUNT,
    "frequency_branch_parameter_count": BRANCH_PARAMETER_COUNT,
    "pytorch_onnx_max_abs_logit_diff": max_abs_diff,
    "host_cpu_ort_batch1_latency_ms": cpu_latency,
    "benchmark_note": (
        "Host CPU model-only diagnostic; excludes crop + luminance + DCT preprocessing."
    ),
    "dct_preprocess": {
        "luminance_weights_rgb": [0.299, 0.587, 0.114],
        "dynamic_range": "sign(C)*log1p(abs(C))",
        "normalization": "per-sample z-score",
        "frequency_mask": "none",
    },
    "calibrated_logit_threshold": float(
        best_meta["calibrated_logit_threshold"]
    ),
    "calibrated_probability_threshold": float(
        best_meta["calibrated_probability_threshold"]
    ),
    "frequency_branch_artifact": BRANCH_BEST_PATH,
}

runtime_config_path = f"{RUN_NAME}_runtime_config.json"
with open(runtime_config_path, "w") as f:
    json.dump(runtime_config, f, indent=2)

print("✅ E2 export complete")
print("ONNX:", onnx_path)
print("Size:", f"{onnx_size_mb:.3f} MiB")
print("Parity max |Δ|:", max_abs_diff)
print("Host CPU ORT latency:", cpu_latency)
print("Runtime config:", runtime_config_path)
print("Frequency branch:", BRANCH_BEST_PATH)


In [ ]:
# ---------------- Final artifact checklist ----------------

expected_outputs = [
    BEST_MODEL_PATH,
    CHECKPOINT_PATH,
    BEST_META_PATH,
    HISTORY_PATH,
    RUN_CONFIG_PATH,
    BRANCH_BEST_PATH,
    BRANCH_SPEC_PATH,
    f"{RUN_NAME}_training_history.png",
    f"{RUN_NAME}_best.onnx",
    f"{RUN_NAME}_runtime_config.json",
]

print("E2 artifacts:")
for path in expected_outputs:
    print(
        "  ",
        "OK " if os.path.exists(path) else "---",
        path,
    )

print("\nKeep the original E1 cache + split manifest too.")
print("E3 must reuse the same manifest/cache policy.")
